# Guichet: retrieval (step 5)
Runtime > Change runtime type > **T4 GPU**. Run cells one at a time and check each output.

In [ ]:
# 1. Mount Drive, get the latest repo, rebuild the chunks
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/guichet-rag || git clone -q https://github.com/AfoDyDigitals/guichet-rag.git /content/guichet-rag
%cd /content/guichet-rag
!git pull -q
!python data/make_chunks.py
!mkdir -p /content/drive/MyDrive/guichet-rag

In [ ]:
# 2. Install libraries (1-2 minutes)
!pip -q install rank_bm25 sentence-transformers

In [ ]:
# 3. Load the models. Check: prints 'cuda'
import json, sys, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
sys.path.insert(0, 'retrieval')
from search import Retriever, load_or_embed, METHODS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
embedder = SentenceTransformer('intfloat/multilingual-e5-small', device=device)
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512, device=device)

In [ ]:
# 4. Embed all chunks (2-5 minutes the first time, seconds after: saved to Drive)
CACHE = '/content/drive/MyDrive/guichet-rag'
retrievers = {}
for name in ['section', 'fixed', 'sentence']:
    chunks = [json.loads(l) for l in open(f'data/chunks_{name}.jsonl')]
    emb = load_or_embed(embedder, chunks, f'{CACHE}/emb_{name}.npy')
    retrievers[name] = Retriever(chunks, embedder, emb, reranker)
    print(name, emb.shape)

In [ ]:
# 5. Smoke test: one of your questions through all 4 methods
q = json.loads(open('evaluation/questions.jsonl').readline())
r = retrievers['section']
print('Q:', q['question'], '| answer:', q['section_id'])
for m in METHODS:
    top = [r.chunks[i] for i in r.search(m, q['question'], 10)]
    hit = any(q['section_id'] in c['section_ids'] for c in top)
    print(f"\n{m:7} answer in top 10: {hit}")
    for c in top[:3]:
        print('   ', c['chunk_id'], c['text'][:90])

## Step 6: score everything

In [ ]:
# 6. Score all question sets x 3 chunkings x 4 methods (5-10 minutes)
import pandas as pd
sys.path.insert(0, 'evaluation')
from evaluate import evaluate, load

question_sets = {'human_en': load('evaluation/questions.jsonl'),
                 'human_fr': load('evaluation/questions_fr.jsonl'),
                 'synthetic': load('evaluation/questions_synthetic.jsonl')}
print({k: len(v) for k, v in question_sets.items()})
rows, ranks = evaluate(retrievers, question_sets, METHODS)
results = pd.DataFrame(rows)
results.to_csv(f'{CACHE}/results.csv', index=False)
with open(f'{CACHE}/ranks.jsonl', 'w') as f:
    f.writelines(json.dumps(x, ensure_ascii=False) + '\n' for x in ranks)
print('saved to Drive')

In [ ]:
# 7. The headline table: recall@5, exact passage found
for qs in question_sets:
    t = results.query('level == "section" and questions == @qs').pivot(index='method', columns='chunking', values='recall@5')
    print(f'\n{qs}  (recall@5, exact passage)')
    print(t.loc[METHODS, ['section', 'fixed', 'sentence']].to_string())

## Experiment: query translation
Run cells 1-5 first (skip 6 and 7), then these two.

In [ ]:
# 8. Translate English questions to French, then score them (2-3 minutes)
!pip -q install sentencepiece sacremoses
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
sys.path.insert(0, 'evaluation')
from evaluate import evaluate, load

name = 'Helsinki-NLP/opus-mt-en-fr'
tokz = AutoTokenizer.from_pretrained(name)
mt = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)

def translate(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = tokz(texts[i:i + bs], return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            ids = mt.generate(**batch, max_new_tokens=128)
        out += tokz.batch_decode(ids, skip_special_tokens=True)
    return out

en = load('evaluation/questions.jsonl')
fr = translate([q['question'] for q in en])
translated = [{**q, 'question': t, 'lang': 'fr', 'translated_from': q['question']} for q, t in zip(en, fr)]
for q in translated[:5]:
    print(q['translated_from'], '\n  ->', q['question'])

rows2, ranks2 = evaluate(retrievers, {'en_translated': translated}, METHODS)
pd.DataFrame(rows2).to_csv(f'{CACHE}/results_translated.csv', index=False)
with open(f'{CACHE}/questions_mt.jsonl', 'w') as f:
    f.writelines(json.dumps(q, ensure_ascii=False) + '\n' for q in translated)
print('saved to Drive')

In [ ]:
# 9. Compare: English vs translated vs French (recall@5, exact passage)
allr = pd.concat([pd.read_csv(f'{CACHE}/results.csv'), pd.read_csv(f'{CACHE}/results_translated.csv')])
for m in ['dense', 'rerank']:
    t = allr.query('level == "section" and method == @m').pivot(index='questions', columns='chunking', values='recall@5')
    print(f'\n{m}')
    print(t.loc[['human_en', 'en_translated', 'human_fr'], ['section', 'fixed', 'sentence']].to_string())

## Step 7: answers
Run cells 1-5 first (skip 6-9), then these two.

In [ ]:
# 10. Load the translation and answer models, try one question (3-5 minutes the first time)
sys.path.insert(0, 'app')
from rag import Translator, Generator, ask
translator = Translator(device=device)
generator = Generator(device=device)
r = retrievers['sentence']
res = ask('Where can I buy a tax stamp?', r, translator, generator)
print('French query:', res['query_fr'], '\n')
for i, p in enumerate(res['passages'], 1):
    print(f'[{i}]', p['chunk_id'], p['text'][:100])
print('\nANSWER:', res['answer'])

In [ ]:
# 11. Answer 50 of your questions and build the review page (about 10 minutes)
import random
from tqdm.auto import tqdm
sys.path.insert(0, 'evaluation')
from evaluate import load
random.seed(0)
sample = random.sample(load('evaluation/questions.jsonl'), 50)
answers = []
for q in tqdm(sample):
    res = ask(q['question'], r, translator, generator)
    answers.append({'question': q['question'], 'section_id': q['section_id'], 'query_fr': res['query_fr'],
                    'answer': res['answer'],
                    'gold_found': any(q['section_id'] in p['section_ids'] for p in res['passages']),
                    'passages': [{'chunk_id': p['chunk_id'], 'text': p['text']} for p in res['passages']]})
with open(f'{CACHE}/answers.jsonl', 'w') as f:
    f.writelines(json.dumps(a, ensure_ascii=False) + '\n' for a in answers)
!python evaluation/make_answer_review.py {CACHE}/answers.jsonl {CACHE}/answer_review.html
print('your passage was among the 5 given to the model:', sum(a['gold_found'] for a in answers), '/ 50')